In [1]:
import glob
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn import pipeline, linear_model, model_selection, metrics, multioutput, svm, preprocessing

In [2]:
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')
target_names = y_test.columns.values

In [3]:
def file_list(X):

    base_directory = 'YOUR/PATH/HERE/*'
    l = []

    for label in X['label']:
        pattern = os.path.join(base_directory, '*', label)
        for file_path in glob.iglob(pattern, recursive=True):
            l.append(file_path)
    return l

In [4]:
def test_fit(X, vectorizer):
    files = file_list(X)
    X = vectorizer.transform((open(filename).read() for filename in files))
    return X

In [5]:
def evalmodel(X_train, X_test, y_train, y_test, target_names, output_file):
    """Train and evaluate model with given features."""
    # can change to other classifiers! 
    model = multioutput.MultiOutputClassifier(svm.LinearSVC(class_weight='balanced', random_state=0))
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions_df = pd.DataFrame(y_pred, columns=target_names)
    predictions_df.to_csv(output_file, index=False)
    scores = pd.DataFrame(metrics.classification_report(y_test, y_pred, target_names=target_names, zero_division=0, output_dict=True)).T
    accuracy = metrics.accuracy_score(y_test,y_pred)
    return model, scores, accuracy

In [6]:
def prediction(X_train, y_train, X_test, y_test, n1, n2, max_words, target_names, output_file):
    vectorizer = TfidfVectorizer(ngram_range = (n1,n2), max_features = max_words, norm = 'l2', sublinear_tf= True)
    fl = file_list(X_train)
    vec = vectorizer.fit_transform((open(filename).read() for filename in fl))
    X_t = test_fit(X_test, vectorizer)
    model, scores, accuracy = evalmodel(vec, X_t, y_train, y_test, target_names, output_file)
    return model, scores, accuracy

In [7]:
prediction(X_train, y_train, X_test, y_test, 1, 2, 5000, target_names, "svm_bigram_10000.csv")

(MultiOutputClassifier(estimator=LinearSVC(class_weight='balanced',
                                           random_state=0)),
                      precision    recall  f1-score  support
 alien point of view   0.666667  0.500000  0.571429      8.0
 belonging             0.500000  0.090909  0.153846     11.0
 father and son        0.818182  0.562500  0.666667     16.0
 friendship            0.250000  0.066667  0.105263     15.0
 human vs. captivity   1.000000  0.181818  0.307692     11.0
 husband and wife      0.090909  0.142857  0.111111      7.0
 infatuation           0.714286  0.588235  0.645161     17.0
 obsession             1.000000  0.166667  0.285714     12.0
 romantic love         0.400000  0.111111  0.173913     18.0
 time travel           1.000000  0.272727  0.428571     11.0
 micro avg             0.583333  0.277778  0.376344    126.0
 macro avg             0.644004  0.268349  0.344937    126.0
 weighted avg          0.648043  0.277778  0.356451    126.0
 samples avg     